## I. Initialization & Environment

In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, clear_output
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR_v2":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR_v2"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR_v2 directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.kernel import FilterPack, EngineInput, MarketObservation, QuantUtils, TickerEngine
from core.config import GLOBAL_SETTINGS, OUTPUT_DIR
from core.environment import ParallelFeatureBuilder, FeatureCubeStitcher, DiscoveryEnv, AlphaLogic
from core.engine import AlphaEngine, AlphaCache
from core.features import generate_features
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_DIR = SU.load_env_from_root("DATA_DIR")
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\nDATA Dir: {DATA_DIR}\nOHLCV: {DATA_PATH_OHLCV}\nIndices: {DATA_PATH_INDICES}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

✓ Paths Initialized.
DATA Dir: c:\Users\ping\Files_win10\python\py311\stocks\data\
OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [2]:
# # === RELOADING FROM PARQUET ===
# features_df = pd.read_parquet("output/features_df.parquet")
# macro_df = pd.read_parquet("output/macro_df.parquet")
# df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
# df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
# df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

# df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
# df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
# df_fed = pd.read_parquet(
#     OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
# )

In [3]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

⏳ Loading DataFrames... takes ~6 minutes
⚡ Generating Features...
⚡ Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
✅ Pipeline Complete. Tickers: 1578, Days: 16192


In [4]:
# === PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

## III. The Analysis Suite

In [5]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [6]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

DEBUG: 941 stocks passed filters on 2026-04-16


In [7]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [8]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=True,
)

print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df)
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
DEBUG: 941 stocks passed filters on 2026-04-16
----------------------------------------------------------------------
Timeline: [2026-04-01] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (3):
SOXL, CRDO, NBIS
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.6818,0.5047,0.1557
Benchmark Gain,0.0858,0.0684,0.0053
== Gain Delta,0.5960,0.4363,0.1504
Group Sharpe,23.1077,24.1712,38.7616
Benchmark Sharpe,10.9495,14.0962,2.3401
== Sharpe Delta,12.1582,10.0750,36.4215
Group Sharpe (ATRP),0.6272,0.6934,0.5230
Benchmark Sharpe (ATRP),0.3892,0.4683,0.0879
== Sharpe (ATRP) Delta,0.2380,0.2251,0.4351



Target Reward Signal: 0.155737


In [9]:
##################################
##################################
##################################

In [10]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 3))
[  4]   📂 tickers (len=3)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]   📈 initial_weights (shape=(3,))
[  9]   📂 perf_metrics (len=24)
[ 10]     🔢 full_p_gain (float)
[ 11]     🔢 full_p_sharpe (float)
[ 12]     🔢 full_p_sharpe_atrp (float)
[ 13]     🔢 full_p_sharpe_trp (float)
[ 14]     🔢 lookback_p_gain (float)
[ 15]     🔢 lookback_p_sharpe (float)
[ 16]     🔢 lookback_p_sharpe_atrp (float)
[ 17]     🔢 lookback_p_sharpe_trp (float)
[ 18]     🔢 holding_p_gain (float)
[ 19]     🔢 holding_p_sharpe (float)
[ 20]     🔢 holding_p_sharpe_atrp (float)
[ 21]     🔢 holding_p_sharpe_trp (float)
[ 22]     🔢 full_b_gain (float)
[ 23]     🔢 full_b_sharpe (float)
[ 24]     🔢 full_b_sharpe_atrp (float)
[ 25]     🔢 full_b_sharpe_trp (float)
[ 26]     🔢 lookback_b_gain (flo

In [11]:
target_tickers = SU.peek(4, result)

 📍 INDEX: [4]
 🏷️  NAME:  tickers
 📂 PATH:  audit_pack -> tickers



['SOXL', 'CRDO', 'NBIS']

In [12]:
SU.export_audit_to_excel(
    audit_pack=analyzer1.last_run, filename="Audit_Verification_Report.xlsx"
)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [13]:
##################################
##################################
##################################

In [14]:
import pathlib

# Set display width to prevent line wrapping
pd.set_option("display.width", 2000)

# Show all columns instead of truncated view
pd.set_option("display.max_columns", None)


def load_latest_finviz_parquet(data_dir: str) -> pd.DataFrame:
    path = pathlib.Path(data_dir)
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"

    files = list(path.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {path.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Usage
df_finviz = load_latest_finviz_parquet(DATA_DIR)
print(f"df_finviz:\n{df_finviz}")

DEBUG: Loading file: 2026-05-04_df_finviz_merged_stocks_etfs.parquet
df_finviz:
       No.                                            Company               Index                  Sector                        Industry Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M    P/E  Fwd P/E   PEG    P/S    P/B    P/C  P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %   ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Pri

In [15]:
df_finviz.loc[target_tickers]

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
SOXL,143,Direxion Daily Semiconductor Bull 3X ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Equity - Leve...",17060.0,802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.06,0.08,9/23/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.14,11.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1207,3.37,141.80,95.63,169.15,859.74,203.47,5.56,8.61,6.7503,8.19,7.13,28.58,75.38,159.83,-2.92,222.75,-2.92,952.39,12.12 - 131.38,-2.92,40893.43,72.24,-,3/11/2010,Yes,Yes,NaN,-3.57,1.43,NaN,86.52,0.53,45976254,NaN,130.40,132.27,133.67,123.80,127.55,-2.19,Equity - Leveraged / Inverse,Equities (Stocks),0.75,43.0,17060.0,NaN,-32.97,NaN,-28.12,-12580.0,-42.44,976.07,115.1,26.07,"U.S., equity, semiconductors, leverage",1.0952,2.4271,1.2162,13.2622,56.8146,8.1580,6.6342,12.6527,2.6700,7.8969,15.1350,3.4915,6.8690,12.3534,2.9727,3.2814,5.0905,1.6996,2.5382,3.8318,1.5078,2.8702,4.5154,1.6255
CRDO,426,Credo Technology Group Holding Ltd,RUT,Technology,Semiconductors,Cayman Islands,NASD,"Technology, Semiconductors",33210.0,505,33210.0,99.86,NaN,0.31,31.09,17.94,25.52,117.07,10.04,7.06,NaN,NaN,-,0.0,1.80,1.02,NaN,NaN,NaN,NaN,NaN,201.49,408.64,6897.08,226.12,1070.0,339.76,17.38,0.21,184.22,165.65,89.92,10.19,-6.37,76.08,3.73,5.58,1.23,9.24,24.65,27.54,18.25,10.82,9.56,0.01,0.01,67.83,30.32,31.81,2.4407,-0.24,77.49,50.10,8.07,273.10,25.14,3.17,11.96,6.6422,7.09,7.49,12.01,39.85,31.13,-9.50,108.20,-15.78,294.44,45.65 - 213.80,-15.78,2402.57,62.33,Mar 02/a,1/27/2022,Yes,Yes,622.0,-3.72,1.43,1.16,7.51,0.48,3623868,212.16,184.38,187.02,188.20,178.88,180.06,-2.34,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,4.8466,17.0583,2.5197,7.5982,26.2272,3.5143,-0.2326,-0.3322,0.9657,2.8823,4.6059,1.5213,5.0519,10.0702,2.2004,2.5178,4.1634,1.4867,0.5366,0.8237,1.0914,1.9902,3.1802,1.3854
NBIS,344,Nebius Group NV,-,Technology,Software - Infrastructure,Netherlands,NASD,"Technology, Software - Infrastructure",44410.0,401,44410.0,346.13,NaN,NaN,83.13,9.68,12.07,NaN,18.23,14.61,NaN,NaN,-,0.0,0.51,-0.77,NaN,NaN,NaN,NaN,NaN,500.79,-70.26,373.88,354.32,534.2,101.60,5.74,-7.92,219.47,201.25,91.70,20.05,-0.26,39.84,1.04,21.68,2.71,43.63,0.25,0.50,1.07,3.08,6.57,1.05,1.06,-7.66,-113.33,19.02,24.9522,21.70,62.12,100.11,42.07,594.57,110.76,3.86,12.33,6.9890,9.26,8.00,17.76,43.18,77.40,4.57,111.21,4.57,658.80,23.25 - 168.71,4.57,1674.83,68.89,May 13/b,5/24/2011,Yes,Yes,NaN,10.12,3.70,1.76,16.07,1.54,24701895,176.85,154.49,160.21,179.40,159.70,176.42,14.20,,,NaN,NaN,NaN,NaN,NaN

In [16]:
print(df_finviz.loc[target_tickers])

      No.                                   Company Index      Sector                   Industry         Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E   PEG    P/S    P/B    P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %   EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %  ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W Low %   

In [17]:
def cleanup_ui():
    clear_output(wait=True)
    pio.renderers.default = None
    gc.collect()
    print("🧹 Memory Cleared.")


def active_engine_audit():
    gc.collect()
    engines = [obj for obj in gc.get_objects() if type(obj).__name__ == "AlphaEngine"]
    print(f"Active AlphaEngine Instances: {len(engines)}")


active_engine_audit()

Active AlphaEngine Instances: 1


In [18]:
# print(features_df.info())
# display(features_df.head())
# display(df_indices.tail())

In [19]:
################################
################################
################################
################################

In [20]:
SU.peek(0, result)

 📍 INDEX: [0]
 🏷️  NAME:  audit_pack
 📂 PATH:  audit_pack



EngineOutput(portfolio_series=Date
2026-04-01    1.0000
2026-04-02    1.0448
2026-04-06    1.0736
2026-04-07    1.1157
2026-04-08    1.2222
2026-04-09    1.2799
2026-04-10    1.3768
2026-04-13    1.4861
2026-04-14    1.6280
2026-04-15    1.6786
2026-04-16    1.6565
2026-04-17    1.6761
2026-04-20    1.7388
2026-04-21    1.7746
2026-04-22    1.8428
2026-04-23    1.8776
2026-04-24    1.9774
dtype: float64, benchmark_series=Date
2026-04-01    1.0000
2026-04-02    1.0009
2026-04-06    1.0056
2026-04-07    1.0061
2026-04-08    1.0317
2026-04-09    1.0377
2026-04-10    1.0370
2026-04-13    1.0471
2026-04-14    1.0599
2026-04-15    1.0682
2026-04-16    1.0708
2026-04-17    1.0838
2026-04-20    1.0816
2026-04-21    1.0745
2026-04-22    1.0854
2026-04-23    1.0812
2026-04-24    1.0896
dtype: float64, normalized_plot_data=Ticker        NBIS    SOXL    CRDO
Date                              
2026-04-01  1.0000  1.0000  1.0000
2026-04-02  1.0674  1.0094  1.0577
2026-04-06  1.1039  1.0488  1.0682
2

EngineOutput(portfolio_series=Date
2026-04-01    1.0000
2026-04-02    1.0448
2026-04-06    1.0736
2026-04-07    1.1157
2026-04-08    1.2222
2026-04-09    1.2799
2026-04-10    1.3768
2026-04-13    1.4861
2026-04-14    1.6280
2026-04-15    1.6786
2026-04-16    1.6565
2026-04-17    1.6761
2026-04-20    1.7388
2026-04-21    1.7746
2026-04-22    1.8428
2026-04-23    1.8776
2026-04-24    1.9774
dtype: float64, benchmark_series=Date
2026-04-01    1.0000
2026-04-02    1.0009
2026-04-06    1.0056
2026-04-07    1.0061
2026-04-08    1.0317
2026-04-09    1.0377
2026-04-10    1.0370
2026-04-13    1.0471
2026-04-14    1.0599
2026-04-15    1.0682
2026-04-16    1.0708
2026-04-17    1.0838
2026-04-20    1.0816
2026-04-21    1.0745
2026-04-22    1.0854
2026-04-23    1.0812
2026-04-24    1.0896
dtype: float64, normalized_plot_data=Ticker        NBIS    SOXL    CRDO
Date                              
2026-04-01  1.0000  1.0000  1.0000
2026-04-02  1.0674  1.0094  1.0577
2026-04-06  1.1039  1.0488  1.0682
2

In [21]:
features_df.loc["TSLA"]

,ATR,ATRP,TRP,RSI,Mom_21,Consistency,IR_63,Beta_63,DD_21,AutoCorr_15,Ret_1d,Range_Pos_20,Slope_P_5,Slope_V_5,Convexity,RollingStalePct,RollMedDollarVol,RollingSameVolCount
Date,,,,,,,,,,,,,,,,,,
2010-06-29,NaN,0.0000,0.0000,50.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-06-30,0.4747,0.2988,0.2988,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.0025,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-01,0.4677,0.3194,0.2573,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.0785,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-02,0.4552,0.3556,0.2286,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.1257,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-06,0.4425,0.4120,0.2588,0.0000,NaN,0.0,NaN,1.0000,0.0000,0.0000,-0.1609,NaN,-0.1004,-0.6060,0.0000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-28,14.6863,0.0391,0.0259,48.7541,0.0392,0.6,-0.1180,1.6338,-0.0614,0.1197,-0.0070,0.5383,-0.0047,0.1559,0.0071,0.0,2.8875e+10,0.0
2026-04-29,14.0916,0.0378,0.0171,47.1592,0.0493,0.4,-0.1258,1.6349,-0.0694,0.1039,-0.0086,0.4936,-0.0006,0.1315,0.0064,0.0,2.8776e+10,0.0
2026-04-30,14.2693,0.0374,0.0434,51.8141,0.0266,0.6,-0.0898,1.6384,-0.0474,0.0782,0.0237,0.6162,0.0013,-0.0814,0.0060,0.0,2.8642e+10,0.0
